# Image Preprocessing Pipeline
Computer Vision - Year 3

In [ ]:
!pip install scikit-image opencv-python-headless -q

import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from skimage import data as skdata
from skimage.color import gray2rgb

In [ ]:
# load the images we're going to use
def load_test_images():
    raw = {
        'astronaut.jpg':           skdata.astronaut(),
        'coffee.jpg':              skdata.coffee(),
        'cat.jpg':                 skdata.cat(),
        'chelsea.jpg':             skdata.chelsea(),
        'rocket.jpg':              skdata.rocket(),
        'hubble_deep_field.jpg':   skdata.hubble_deep_field(),
        'immunohistochemistry.jpg':skdata.immunohistochemistry(),
        'retina.jpg':              skdata.retina(),
        'colorwheel.jpg':          skdata.colorwheel(),
        'grass.jpg':               skdata.grass(),
        'gravel.jpg':              skdata.gravel(),
        'brick.jpg':               skdata.brick(),
        'clock.jpg':               skdata.clock(),
        'moon.jpg':                skdata.moon(),
        'camera.jpg':              skdata.camera(),
        'coins.jpg':               skdata.coins(),
        'horse.jpg':               skdata.horse(),
        'page.jpg':                skdata.page(),
        'text.jpg':                skdata.text(),
        'logo.jpg':                skdata.logo(),
    }

    result = {}
    for name, img in raw.items():
        if img.dtype == bool:
            img = img.astype(np.uint8) * 255
        if img.dtype != np.uint8:
            img = (img * 255).astype(np.uint8)
        if img.ndim == 2:
            img = gray2rgb(img)
        if img.ndim == 3 and img.shape[2] == 4:
            img = img[:, :, :3]
        result[name] = img

    return result

images = load_test_images()
print(f'loaded {len(images)} images')

In [ ]:
# the main processing function
# takes an image and applies 3 steps: blur, clahe, sharpen
def process_image_array(img_rgb):
    original = img_rgb.copy()

    # gaussian blur to reduce noise
    denoised = cv2.GaussianBlur(img_rgb, (3, 3), sigmaX=1)

    # clahe to improve contrast (working on L channel only)
    lab = cv2.cvtColor(denoised, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    enhanced = cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2RGB)

    # sharpen using unsharp masking
    blur = cv2.GaussianBlur(enhanced, (0, 0), sigmaX=2)
    final = cv2.addWeighted(enhanced, 1.5, blur, -0.5, 0)

    return original, denoised, enhanced, final

In [ ]:
# show all 4 stages side by side
def show_pipeline(original, denoised, enhanced, final, title=''):
    stages = [
        (original, 'Original',  'no changes'),
        (denoised, 'Denoised',  'gaussian blur 3x3'),
        (enhanced, 'Enhanced',  'CLAHE'),
        (final,    'Final',     'unsharp masking'),
    ]

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    for ax, (img, name, method) in zip(axes, stages):
        ax.imshow(img)
        ax.set_title(f'{name}\n{method}', fontsize=10)
        ax.axis('off')

    if title:
        plt.suptitle(title, fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()


# before / after only
def show_comparison(original, final, title=''):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(original)
    axes[0].set_title('Before')
    axes[0].axis('off')
    axes[1].imshow(final)
    axes[1].set_title('After')
    axes[1].axis('off')
    if title:
        plt.suptitle(title, fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()


# histogram to compare brightness before and after clahe
def show_histogram(original, final, title=''):
    def get_l(img):
        return cv2.split(cv2.cvtColor(img, cv2.COLOR_RGB2LAB))[0]

    l_before = get_l(original)
    l_after  = get_l(final)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(l_before.ravel(), bins=128, color='gray', alpha=0.8)
    axes[0].set_title('before')
    axes[0].set_xlim(0, 255)
    axes[0].axvline(l_before.mean(), color='black', linestyle='--')

    axes[1].hist(l_after.ravel(), bins=128, color='green', alpha=0.8)
    axes[1].set_title('after CLAHE')
    axes[1].set_xlim(0, 255)
    axes[1].axvline(l_after.mean(), color='black', linestyle='--')

    if title:
        plt.suptitle(title, y=1.04)
    plt.tight_layout()
    plt.show()


# grid showing n images, each row = one image with its 4 stages
def show_comparison_grid(results_list, n=4):
    samples = results_list[:n]
    if not samples:
        print('nothing to show')
        return

    fig = plt.figure(figsize=(20, 5 * len(samples)))
    col_labels = ['Original', 'Denoised', 'Enhanced', 'Final']

    for row, (name, orig, den, enh, fin) in enumerate(samples):
        for col, img in enumerate([orig, den, enh, fin]):
            ax = fig.add_subplot(len(samples), 4, row * 4 + col + 1)
            ax.imshow(img)
            ax.axis('off')
            if row == 0:
                ax.set_title(col_labels[col], fontsize=11)
            if col == 0:
                ax.set_ylabel(os.path.basename(name), fontsize=9, rotation=0, labelpad=80, va='center')

    plt.suptitle('results', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
# run everything
results_list = []

for img_name, img_array in images.items():
    print(f'processing: {img_name}')
    original, denoised, enhanced, final = process_image_array(img_array)
    show_pipeline(original, denoised, enhanced, final, title=img_name)
    show_histogram(original, final, title=img_name)
    results_list.append((img_name, original, denoised, enhanced, final))

show_comparison_grid(results_list, n=4)
print('done')